# Baseline zero-shot (V1)

Primer baseline del proyecto: inferencia directa de los 4 modelos sin ningún tipo
de fine-tuning. Sirve como línea base para medir el impacto real de las fases posteriores.

**Modelos evaluados:**
- **Flan-T5-base** — encoder-decoder clásico, instruction-tuned
- **Qwen3-1.7B** — decoder-only moderno, generalista
- **BART-large-cnn** — referencia (ya fine-tuneado en CNN/DailyMail)
- **Pegasus-cnn_dailymail** — referencia (diseñado específicamente para summarization)

Los dos últimos actúan como "techo" de referencia — son el estado del arte específico
de la tarea y nos permiten contextualizar los resultados propios.

In [4]:
# Setup
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import pandas as pd

from src.data.loader import load_config, load_cnn_dailymail
from src.models.loader import load_model, free_model
from src.evaluation.inference import generate_summaries
from src.evaluation.metrics import compute_rouge
# Silence non-actionable warnings for a cleaner notebook output
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

RESULTS_DIR = Path("../results/tables")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CUDA available: True
Device: NVIDIA GeForce RTX 5070


In [5]:
# Load config and test subset
cfg = load_config("../config/config.yaml")
dataset = load_cnn_dailymail(cfg)

# Use a manageable subset for the zero-shot baseline (not the full test set yet)
N_EVAL = 200
test_subset = dataset["test"].select(range(N_EVAL))
articles = test_subset["article"]
references = test_subset["highlights"]

print(f"Evaluating on {len(articles)} test articles")

Evaluating on 200 test articles


In [6]:
# Iterate over models one at a time to respect VRAM budget
models_to_eval = ["bart_reference", "pegasus_reference", "t5", "qwen"]

all_results = {}
all_predictions = {}

for model_key in models_to_eval:
    print(f"\n{'='*60}\nLoading {model_key}\n{'='*60}")
    loaded = load_model(cfg["models"][model_key])

    start = time.time()
    preds = generate_summaries(
        loaded,
        articles,
        max_new_tokens=cfg["dataset"]["max_target_length"],
        num_beams=cfg["generation"]["num_beams"],
        batch_size=4,
    )
    elapsed = time.time() - start

    scores = compute_rouge(preds, references)
    scores["seconds"] = round(elapsed, 1)
    scores["sec_per_sample"] = round(elapsed / len(articles), 2)

    all_results[model_key] = scores
    all_predictions[model_key] = preds

    print(f"Results: {scores}")
    free_model(loaded)


Loading bart_reference


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Generating [facebook/bart-large-cnn]:   0%|          | 0/50 [00:00<?, ?it/s]

Results: {'rouge1': np.float64(35.14), 'rouge2': np.float64(14.54), 'rougeL': np.float64(25.52), 'rougeLsum': np.float64(29.25), 'seconds': 51.9, 'sec_per_sample': 0.26}

Loading pegasus_reference


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

...

Generating [google/flan-t5-base]:   0%|          | 0/50 [00:00<?, ?it/s]

Results: {'rouge1': np.float64(24.96), 'rouge2': np.float64(9.03), 'rougeL': np.float64(19.17), 'rougeLsum': np.float64(21.64), 'seconds': 32.4, 'sec_per_sample': 0.16}

Loading qwen


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Generating [Qwen/Qwen3-1.7B]:   0%|          | 0/50 [00:00<?, ?it/s]

Results: {'rouge1': np.float64(22.63), 'rouge2': np.float64(5.33), 'rougeL': np.float64(14.84), 'rougeLsum': np.float64(18.23), 'seconds': 648.9, 'sec_per_sample': 3.24}


In [7]:
# Results summary table
results_df = pd.DataFrame(all_results).T
results_df.index.name = "model"
results_df = results_df[["rouge1", "rouge2", "rougeL", "rougeLsum", "seconds", "sec_per_sample"]]
results_df.to_csv(RESULTS_DIR / "v1_zero_shot_baseline.csv")
results_df

,rouge1,rouge2,rougeL,rougeLsum,seconds,sec_per_sample
model,,,,,,
bart_reference,35.14,14.54,25.52,29.25,51.9,0.26
pegasus_reference,35.07,14.38,25.73,31.76,69.6,0.35
t5,24.96,9.03,19.17,21.64,32.4,0.16
qwen,22.63,5.33,14.84,18.23,648.9,3.24


In [8]:
# Qualitative inspection: show one example per model
example_idx = 0
print("=" * 80)
print(f"ARTICLE (truncated):\n{articles[example_idx][:400]}...\n")
print(f"REFERENCE:\n{references[example_idx]}\n")
print("=" * 80)
for model_key, preds in all_predictions.items():
    print(f"\n[{model_key}]")
    print(preds[example_idx])

ARTICLE (truncated):
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also acce...

REFERENCE:
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


[bart_reference]
The Palestinian Authority becomes the 123rd member of the International Criminal Court. The move gives the court jurisdiction over alleged crimes in Palestinian territories. Israel and the United States opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki said it 

## Análisis del baseline V1

Los resultados zero-shot establecen una jerarquía clara y coherente:

| Modelo                | ROUGE-1 | ROUGE-2 | ROUGE-L | ROUGE-Lsum | s/muestra |
|-----------------------|---------|---------|---------|------------|-----------|
| BART-large-cnn        | 35.14   | 14.54   | 25.52   | 29.25      | 0.26      |
| Pegasus-cnn_dailymail | 35.07   | 14.38   | 25.73   | 31.76      | 0.35      |
| Flan-T5-base          | 24.96   | 9.03    | 19.17   | 21.64      | 0.16      |
| Qwen3-1.7B            | 22.63   | 5.33    | 14.84   | 18.23      | 3.24      |

**Techos de referencia (BART, Pegasus).** Ambos modelos están específicamente
fine-tuneados sobre CNN/DailyMail y marcan el estado del arte práctico para esta
tarea. Pegasus iguala a BART en ROUGE-1 y le supera en ROUGE-Lsum (31.76 vs 29.25),
consistente con su objetivo de pre-entrenamiento Gap-Sentence Generation, diseñado
para generar oraciones completas y bien estructuradas.

**Flan-T5-base (24.96 ROUGE-1).** A ~10 puntos del techo, su handicap principal no
es tanto la capacidad del modelo sino la ventana de contexto: con
`max_input_length=512` truncamos el 73% de los artículos, privándole de información
crítica para resumir. Pese a ello, es también el **modelo más rápido** del baseline
(0.16 s/muestra), lo que lo convierte en un candidato interesante a la hora de
ponderar calidad contra coste de inferencia. Este gap respecto al techo es el que
V2 (fine-tuning) intentará cerrar.

**Qwen3-1.7B (22.63 ROUGE-1).** El decoder-only moderno zero-shot sobre una tarea
para la que no fue entrenado específicamente. Es notable que con 1.7B de parámetros
(casi 7x más que Flan-T5-base) se quede apenas 2.4 puntos por debajo de este en
ROUGE-1, y que caiga a 5.3 en ROUGE-2 — patrón clásico de LLMs generalistas en
summarization: entienden qué decir (vocabulario relevante) pero parafrasean en
lugar de reproducir n-gramas literales del estilo periodístico. Aquí es donde LoRA
fine-tuning tiene el mayor margen de mejora potencial.

**Coste computacional.** Qwen3 es aproximadamente 18x más lento en inferencia que
Flan-T5-base (3.24 s/muestra vs 0.16) y ~10x más lento que BART/Pegasus. El coste
se debe a: (1) generación token-a-token en decoder-only, (2) secuencia de entrada
3x más larga (1536 vs 512 tokens), y (3) beam search. Este será un eje importante
del análisis de eficiencia comparado.

**Nota metodológica.** Los scores de BART y Pegasus son ~9 puntos inferiores a los
publicados en los papers originales (~44 ROUGE-1). Este gap es esperable y se debe a:
(1) evaluación sobre un subset de 200 muestras frente a las ~11.500 del test completo,
(2) diferencias en el post-procesado específico de los papers, y (3) sensibilidad
conocida de ROUGE a detalles de implementación. La comparación **relativa** entre los
cuatro modelos sigue siendo metodológicamente válida y refleja correctamente la
jerarquía esperada.

Estas cifras son el punto de partida que intentaremos superar en V2 (fine-tuning básico),
V3 (optimización de hiperparámetros) y V4 (modelo final + quantización).
